# Regression Models — Basic
**Prerequisites:** Python basics, familiarity with NumPy and Pandas.

This notebook introduces regression in machine learning using three fundamental linear models.
After completing this notebook, move on to **Regression Models — Advanced** for tree-based
methods and hyperparameter tuning.

## What is Regression?

Regression is a type of supervised learning where the goal is to predict a **continuous numerical value**
from one or more input features.

Examples:
- Predicting house prices from square footage and location
- Estimating a patient's blood pressure from health metrics
- Forecasting tomorrow's temperature from today's weather data

The model learns a mapping function $f(X) \rightarrow y$ from labelled training examples,
then applies it to unseen data.

### Key Terms
- **Feature (X):** An input variable used to make predictions
- **Target (y):** The continuous value we want to predict
- **Training:** Fitting the model to historical data
- **Generalization:** How well the model performs on new, unseen data

## Dataset: Boston Housing

We'll use the Boston Housing dataset — a classic benchmark with 506 samples and 13 features describing neighbourhoods in Boston. The target is `MEDV`: the median house value in $1,000s.

> **Ethical Note:** This dataset has been deprecated from scikit-learn (v1.2+) because the `B` feature reflects problematic racial assumptions. We use it here for educational purposes only. For new projects, prefer `sklearn.datasets.fetch_california_housing()`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## Loading the Data

In [ ]:
column_names = [
    'CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX',
    'PTRATIO', 'B', 'LSTAT', 'MEDV'
]

df = pd.read_csv('../data/housing.csv', header=None, delimiter=r"\s+", names=column_names)
print(f"Dataset shape: {df.shape}")
df.sample(5)

## Exploring the Data

Before building any model, always understand what you're working with.

In [ ]:
df.describe()

In [ ]:
# Check for missing values
df.isnull().sum()

In [ ]:
# Distribution of our target variable
plt.figure(figsize=(7, 4))
sns.histplot(df['MEDV'], bins=30, kde=True, color='steelblue')
plt.title('Distribution of MEDV (Median House Value)')
plt.xlabel('MEDV ($1000s)')
plt.ylabel('Count')
plt.show()

# Note: The spike at 50 suggests the dataset caps house values at $50k

In [ ]:
# Which features correlate most with house price?
corr = df.corr()['MEDV'].drop('MEDV').sort_values()

plt.figure(figsize=(8, 5))
corr.plot(kind='barh', color=['tomato' if v < 0 else 'steelblue' for v in corr])
plt.title('Feature Correlation with MEDV')
plt.xlabel('Pearson Correlation')
plt.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

In [ ]:
# RM (avg rooms) vs price — the strongest positive relationship
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(df['RM'], df['MEDV'], alpha=0.4, color='steelblue')
axes[0].set_xlabel('Avg Rooms per Dwelling (RM)')
axes[0].set_ylabel('Median Value (MEDV)')
axes[0].set_title('RM vs MEDV')

axes[1].scatter(df['LSTAT'], df['MEDV'], alpha=0.4, color='tomato')
axes[1].set_xlabel('% Lower Status Population (LSTAT)')
axes[1].set_ylabel('Median Value (MEDV)')
axes[1].set_title('LSTAT vs MEDV')

plt.tight_layout()
plt.show()

## Train-Test Split

We split the data into a **training set** (80%) and a **test set** (20%).

- **Training set:** The model learns from this data
- **Test set:** Held out completely — used only to evaluate final performance

Always split *before* any preprocessing to avoid **data leakage**.

In [ ]:
X = df.drop('MEDV', axis=1)
y = df['MEDV']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples:     {len(X_test)}")

## Feature Scaling

**StandardScaler** transforms each feature to have mean = 0 and standard deviation = 1:

$$X_{scaled} = \frac{X - \mu}{\sigma}$$

This is important for **Linear Regression with regularization** (Ridge, Lasso) and for distance-based models.
Tree-based models (covered in the Advanced notebook) don't need scaling.

**Key rule:** Always fit the scaler on training data only, then apply to both train and test.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # Use transform only — never fit on test data!

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)
print("Scaling done. Train mean (should be ≈0):", X_train_scaled.mean().round(2).head())

## Evaluation Metrics

Before building models, let's understand how we'll measure their performance.

### Mean Absolute Error (MAE)
$$MAE = \frac{1}{n} \sum |y_i - \hat{y}_i|$$
Average absolute error. Easy to interpret — same units as the target.

### Root Mean Squared Error (RMSE)
$$RMSE = \sqrt{\frac{1}{n} \sum (y_i - \hat{y}_i)^2}$$
Like MAE but penalises large errors more heavily. Also in target units.

### R² (Coefficient of Determination)
$$R^2 = 1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$$
Proportion of variance explained. R² = 1 is perfect; R² = 0 means no better than always predicting the mean.

In [ ]:
def evaluate(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f"{'─'*40}")
    print(f"  {name}")
    print(f"  MAE:  {mae:.3f}")
    print(f"  RMSE: {rmse:.3f}")
    print(f"  R²:   {r2:.4f}")

## Linear Regression

The simplest regression model. It fits a straight line (or hyperplane in multiple dimensions)
by minimising the sum of squared errors between predictions and actual values.

$$\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_n x_n$$

**Strengths:** Fast, interpretable, great baseline.  
**Weaknesses:** Assumes a linear relationship; sensitive to outliers and multicollinearity.

In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

evaluate("Linear Regression", y_test, y_pred_lr)

# Coefficients tell us which features matter most
coef_df = pd.Series(lr.coef_, index=X.columns).sort_values()
plt.figure(figsize=(8, 4))
coef_df.plot(kind='barh', color=['tomato' if v < 0 else 'steelblue' for v in coef_df])
plt.title('Linear Regression — Feature Coefficients')
plt.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

## Ridge Regression (L2 Regularization)

Ridge adds a penalty to the loss function that discourages large coefficients:

$$\text{Loss} = \sum(y_i - \hat{y}_i)^2 + \alpha \sum \beta_j^2$$

The `alpha` parameter controls regularization strength:
- **alpha = 0:** Same as Linear Regression (no penalty)
- **Large alpha:** Stronger regularization — coefficients shrink toward zero
- Coefficients shrink but **never reach exactly zero** (no feature elimination)

Ridge is especially useful when features are **highly correlated** (multicollinearity).

In [ ]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)
y_pred_ridge = ridge.predict(X_test_scaled)

evaluate("Ridge Regression (alpha=1.0)", y_test, y_pred_ridge)

In [ ]:
# How does alpha affect performance?
alphas = [0.01, 0.1, 1.0, 10, 100]
ridge_r2 = []
for a in alphas:
    m = Ridge(alpha=a).fit(X_train_scaled, y_train)
    ridge_r2.append(r2_score(y_test, m.predict(X_test_scaled)))

plt.figure(figsize=(7, 4))
plt.semilogx(alphas, ridge_r2, 'o-', color='steelblue')
plt.xlabel('Alpha (log scale)')
plt.ylabel('R² on Test Set')
plt.title('Ridge Regression — Effect of Alpha')
plt.grid(True)
plt.show()

## Lasso Regression (L1 Regularization)

Lasso uses the absolute value of coefficients as its penalty:

$$\text{Loss} = \sum(y_i - \hat{y}_i)^2 + \alpha \sum |\beta_j|$$

The key difference from Ridge: Lasso can drive coefficients to **exactly zero**, effectively
removing features from the model. This makes Lasso useful for **automatic feature selection**.

In [ ]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=1.0)
lasso.fit(X_train_scaled, y_train)
y_pred_lasso = lasso.predict(X_test_scaled)

evaluate("Lasso Regression (alpha=1.0)", y_test, y_pred_lasso)

# Show which features were zeroed out
coef_lasso = pd.Series(lasso.coef_, index=X.columns)
zeroed = coef_lasso[coef_lasso == 0].index.tolist()
print(f"\nFeatures eliminated by Lasso: {zeroed if zeroed else 'None at alpha=1.0'}")

In [ ]:
# Compare Ridge vs Lasso coefficients side-by-side
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

pd.Series(ridge.coef_, index=X.columns).plot(
    kind='barh', ax=axes[0], color='steelblue', title='Ridge Coefficients')
axes[0].axvline(0, color='black', linewidth=0.8)

pd.Series(lasso.coef_, index=X.columns).plot(
    kind='barh', ax=axes[1], color='tomato', title='Lasso Coefficients')
axes[1].axvline(0, color='black', linewidth=0.8)

plt.suptitle('Ridge vs Lasso — Coefficient Comparison', fontsize=13)
plt.tight_layout()
plt.show()

## Comparison: Linear, Ridge, and Lasso

Let's compare our three basic models side-by-side.

In [ ]:
import time

basic_models = {
    'Linear Regression': (LinearRegression(), X_train, X_test),
    'Ridge (α=1)':       (Ridge(alpha=1.0),  X_train_scaled, X_test_scaled),
    'Lasso (α=1)':       (Lasso(alpha=1.0),  X_train_scaled, X_test_scaled),
}

results = []
for name, (model, Xtr, Xte) in basic_models.items():
    model.fit(Xtr, y_train)
    pred = model.predict(Xte)
    results.append({
        'Model': name,
        'MAE':  round(mean_absolute_error(y_test, pred), 3),
        'RMSE': round(np.sqrt(mean_squared_error(y_test, pred)), 3),
        'R²':   round(r2_score(y_test, pred), 4),
    })

results_df = pd.DataFrame(results).set_index('Model')
print(results_df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

results_df[['MAE', 'RMSE']].plot(kind='bar', ax=axes[0], rot=15)
axes[0].set_title('Error Metrics (lower is better)')
axes[0].set_ylabel('Score')
axes[0].grid(axis='y')

results_df[['R²']].plot(kind='bar', ax=axes[1], color='steelblue', rot=15, legend=False)
axes[1].set_title('R² Score (higher is better)')
axes[1].set_ylabel('R²')
axes[1].set_ylim(0, 1)
axes[1].grid(axis='y')

plt.suptitle('Basic Regression Models — Comparison', fontsize=13)
plt.tight_layout()
plt.show()

## Summary

| Model | Key Idea | Feature Selection | Needs Scaling |
|-------|----------|-------------------|---------------|
| Linear Regression | Minimise squared error | No | Optional |
| Ridge (L2) | Penalise large coefficients | No (shrinks) | Yes |
| Lasso (L1) | Penalise absolute coefficients | Yes (zeroes out) | Yes |

**Next steps:** Head to **Regression Models — Advanced** to learn ElasticNet, Support Vector Regression, Decision Trees, Random Forests, Gradient Boosting, and hyperparameter tuning.